# Wordle Solver: Comparative Evaluation

This notebook trains and evaluates four different Wordle-solving strategies:

1. **Frequency Heuristic** — Letter frequency + positional scoring
2. **Information Gain** — Entropy-based optimal guess selection
3. **DQN (Deep Q-Network)** — Value-based reinforcement learning
4. **A2C (Advantage Actor-Critic)** — Policy-gradient reinforcement learning

**Benchmark:** Bertsimas & Paskov (2024) optimal solution: 3.421 avg guesses, 100% win rate

### References
1. Bhambri, S., Bhattacharjee, A., & Bertsekas, D. (2022). *RL Methods for Wordle: A POMDP/Adaptive Control Approach.* arXiv:2211.10298.
2. Liu, C.-L. (2022). *Using Wordle for Learning to Design and Compare Strategies.* IEEE CoG.
3. Bertsimas, D. & Paskov, A. (2024). *An Exact and Interpretable Solution to Wordle.* Operations Research.
4. Anderson, B.J. & Meyer, J.G. (2022). *Finding the optimal human strategy for Wordle.* arXiv:2202.00557.
5. Ho, A. (2022). *Wordle Solving with Deep Reinforcement Learning.*

## Setup

In [ ]:
# If running in Google Colab, clone the repo first:
# !git clone https://github.com/jhffmn82/wordle-ML-project.git
# %cd wordle-ML-project

# Install dependencies
# !pip install torch numpy matplotlib tqdm

In [ ]:
import sys
import os
import time
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Add project root to path
sys.path.insert(0, os.getcwd())

from engine.wordle_env import WordleGame, load_word_list, get_feedback, filter_words
from engine.state_encoder import encode_state, encode_words_onehot, STATE_DIM
from solvers.frequency_solver import FrequencySolver
from solvers.infogain_solver import InfoGainSolver
from solvers.dqn_solver import DQNNetwork, DQNSolver, ReplayBuffer
from solvers.a2c_solver import A2CNetwork, A2CSolver, RolloutStorage

# Load word list
WORDS = load_word_list()
print(f"Loaded {len(WORDS)} words")

# Pre-compute word encodings (shared by learned models)
WORD_ENCODINGS = torch.tensor(encode_words_onehot(WORDS), dtype=torch.float32)
print(f"Word encodings shape: {WORD_ENCODINGS.shape}")

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Part 1: Heuristic Solvers

These solvers require no training — they compute the best guess on-the-fly using
mathematical scoring of the remaining word list.

### 1.1 Frequency Heuristic

Scores each word by how common its letters are among the remaining candidates,
with a 3x bonus for positional frequency. Simple but effective.

In [ ]:
# Quick demo of the frequency solver
from solvers.frequency_solver import play_game as freq_play

freq_solver = FrequencySolver()
demo_words = ["crane", "slink", "nymph", "fuzzy", "vivid"]

for word in demo_words:
    print(f"\nTarget: {word.upper()}")
    freq_play(freq_solver, word, verbose=True)

### 1.2 Information Gain Solver

For each candidate guess, simulates feedback against every remaining word.
Selects the guess that minimizes the expected remaining word list size.
More computationally expensive but produces better results.

In [ ]:
# Quick demo of the info gain solver
from solvers.infogain_solver import play_game as ig_play

ig_solver = InfoGainSolver()

for word in demo_words:
    print(f"\nTarget: {word.upper()}")
    ig_play(ig_solver, word, verbose=True)

---
## Part 2: Training the DQN Model

The Deep Q-Network learns a function Q(state, action) that estimates how good
each guess is in a given board state. We train by playing thousands of games
and updating the network based on rewards.

**Reference:** Anderson, B.J. & Meyer, J.G. (2022). arXiv:2202.00557.

In [ ]:
def compute_reward(feedback, solved, failed):
    """
    Compute reward for a single guess.
    
    Reward structure (from Anderson & Meyer, 2022):
        Green letter:  +5
        Yellow letter: +2
        Gray letter:    0
        Winning:      +25
        Losing:       -15
    """
    reward = 0.0
    for fb in feedback:
        if fb == 2:    # green
            reward += 5.0
        elif fb == 1:  # yellow
            reward += 2.0
    
    if solved:
        reward += 25.0
    elif failed:
        reward -= 15.0
    
    return reward

In [ ]:
def train_dqn(num_episodes=50000, batch_size=64, gamma=0.95,
              epsilon_start=0.9, epsilon_end=0.1, epsilon_decay=0.99995,
              lr=0.002, target_update=500, log_interval=1000,
              curriculum_sizes=[100, 500, None]):
    """
    Train a DQN model to play Wordle.
    
    Uses bootstrapped training: starts with a small word subset,
    then scales to the full vocabulary.
    
    Args:
        num_episodes: total training episodes
        curriculum_sizes: list of word subset sizes (None = full list)
    
    Returns:
        trained DQNNetwork, training history dict
    """
    # Initialize networks
    policy_net = DQNNetwork().to(device)
    target_net = DQNNetwork().to(device)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()
    
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    replay_buffer = ReplayBuffer(capacity=100000)
    word_enc = WORD_ENCODINGS.to(device)
    
    # Training history
    history = {"episode": [], "reward": [], "win_rate": [], "avg_guesses": [], "epsilon": []}
    recent_wins = []
    recent_guesses = []
    epsilon = epsilon_start
    
    # Curriculum: divide episodes across word subset sizes
    episodes_per_phase = num_episodes // len(curriculum_sizes)
    
    for phase, subset_size in enumerate(curriculum_sizes):
        if subset_size is None:
            train_words = WORDS
        else:
            train_words = random.sample(WORDS, min(subset_size, len(WORDS)))
        
        phase_start = phase * episodes_per_phase
        phase_end = (phase + 1) * episodes_per_phase if phase < len(curriculum_sizes) - 1 else num_episodes
        
        print(f"\n--- Phase {phase+1}: Training on {len(train_words)} words "
              f"(episodes {phase_start}-{phase_end}) ---")
        
        for episode in range(phase_start, phase_end):
            # Pick a target word
            target = random.choice(train_words)
            game = WordleGame(target=target, word_list=WORDS)
            
            guesses = []
            feedbacks = []
            episode_reward = 0
            
            while not game.is_over():
                turn = game.turn
                state = encode_state(guesses, feedbacks, turn)
                state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                
                # Epsilon-greedy action selection
                if random.random() < epsilon:
                    # Random exploration
                    action_idx = random.randint(0, len(WORDS) - 1)
                else:
                    # Greedy exploitation
                    with torch.no_grad():
                        output = policy_net(state_tensor)
                        scores = torch.matmul(output, word_enc.T).squeeze(0)
                        action_idx = scores.argmax().item()
                
                guess = WORDS[action_idx]
                feedback = game.make_guess(guess)
                guesses.append(guess)
                feedbacks.append(feedback)
                
                solved = game.is_solved()
                failed = game.turn >= 6 and not solved
                reward = compute_reward(feedback, solved, failed)
                episode_reward += reward
                
                next_state = encode_state(guesses, feedbacks, game.turn)
                done = solved or failed
                
                replay_buffer.push(state, action_idx, reward, next_state, done)
                
                # Train on a batch
                if len(replay_buffer) >= batch_size:
                    states_b, actions_b, rewards_b, next_states_b, dones_b = \
                        replay_buffer.sample(batch_size)
                    
                    states_b = states_b.to(device)
                    actions_b = actions_b.to(device)
                    rewards_b = rewards_b.to(device)
                    next_states_b = next_states_b.to(device)
                    dones_b = dones_b.to(device)
                    
                    # Current Q-values
                    output = policy_net(states_b)
                    all_q = torch.matmul(output, word_enc.T)
                    q_values = all_q.gather(1, actions_b.unsqueeze(1)).squeeze(1)
                    
                    # Target Q-values
                    with torch.no_grad():
                        next_output = target_net(next_states_b)
                        next_q = torch.matmul(next_output, word_enc.T)
                        max_next_q = next_q.max(dim=1)[0]
                        target_q = rewards_b + gamma * max_next_q * (1 - dones_b)
                    
                    loss = F.mse_loss(q_values, target_q)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                
                if done:
                    break
            
            # Decay epsilon
            epsilon = max(epsilon_end, epsilon * epsilon_decay)
            
            # Update target network
            if episode % target_update == 0:
                target_net.load_state_dict(policy_net.state_dict())
            
            # Track stats
            recent_wins.append(1 if game.is_solved() else 0)
            recent_guesses.append(game.turn if game.is_solved() else 7)
            if len(recent_wins) > 1000:
                recent_wins.pop(0)
                recent_guesses.pop(0)
            
            if (episode + 1) % log_interval == 0:
                win_rate = sum(recent_wins) / len(recent_wins)
                avg_guess = sum(recent_guesses) / len(recent_guesses)
                history["episode"].append(episode + 1)
                history["reward"].append(episode_reward)
                history["win_rate"].append(win_rate)
                history["avg_guesses"].append(avg_guess)
                history["epsilon"].append(epsilon)
                print(f"  Episode {episode+1}: win_rate={win_rate:.3f}, "
                      f"avg_guesses={avg_guess:.2f}, epsilon={epsilon:.3f}")
    
    return policy_net, history

In [ ]:
# Train DQN
# Note: On CPU this may take 1-2 hours. On GPU (Colab), ~30 min.
# Reduce num_episodes for a quick test run.

print("Training DQN...")
start_time = time.time()

dqn_model, dqn_history = train_dqn(
    num_episodes=30000,       # Increase for better results (50k-100k)
    batch_size=64,
    gamma=0.95,
    epsilon_start=0.9,
    epsilon_end=0.1,
    epsilon_decay=0.99995,
    lr=0.002,
    target_update=500,
    log_interval=1000,
    curriculum_sizes=[100, 500, None]
)

elapsed = time.time() - start_time
print(f"\nDQN training complete in {elapsed/60:.1f} minutes")

# Save model
os.makedirs("models", exist_ok=True)
torch.save(dqn_model.state_dict(), "models/dqn_model.pt")
print("Saved to models/dqn_model.pt")

In [ ]:
# Plot DQN training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(dqn_history["episode"], dqn_history["win_rate"])
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Win Rate")
axes[0].set_title("DQN Win Rate")
axes[0].set_ylim(0, 1)
axes[0].axhline(y=1.0, color='r', linestyle='--', alpha=0.3, label='Optimal')
axes[0].legend()

axes[1].plot(dqn_history["episode"], dqn_history["avg_guesses"])
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Avg Guesses")
axes[1].set_title("DQN Avg Guesses")
axes[1].axhline(y=3.421, color='r', linestyle='--', alpha=0.3, label='Optimal (3.421)')
axes[1].legend()

axes[2].plot(dqn_history["episode"], dqn_history["epsilon"])
axes[2].set_xlabel("Episode")
axes[2].set_ylabel("Epsilon")
axes[2].set_title("DQN Exploration Rate")

plt.tight_layout()
plt.savefig("dqn_training.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Part 3: Training the A2C Model

The Advantage Actor-Critic uses two networks:
- **Actor** picks the action (probability distribution over words)
- **Critic** evaluates how good the current state is

The advantage signal (actual return - critic estimate) trains the actor.

**Reference:** Bhambri, S. et al. (2022). arXiv:2211.10298.

In [ ]:
def train_a2c(num_episodes=50000, gamma=0.99, lr=0.001,
              entropy_coef=0.01, value_coef=0.5,
              log_interval=1000, curriculum_sizes=[100, 500, None]):
    """
    Train an A2C model to play Wordle.
    
    Returns:
        trained A2CNetwork, training history dict
    """
    model = A2CNetwork().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    word_enc = WORD_ENCODINGS.to(device)
    
    history = {"episode": [], "reward": [], "win_rate": [], "avg_guesses": []}
    recent_wins = []
    recent_guesses = []
    
    episodes_per_phase = num_episodes // len(curriculum_sizes)
    
    for phase, subset_size in enumerate(curriculum_sizes):
        if subset_size is None:
            train_words = WORDS
        else:
            train_words = random.sample(WORDS, min(subset_size, len(WORDS)))
        
        phase_start = phase * episodes_per_phase
        phase_end = (phase + 1) * episodes_per_phase if phase < len(curriculum_sizes) - 1 else num_episodes
        
        print(f"\n--- Phase {phase+1}: Training on {len(train_words)} words "
              f"(episodes {phase_start}-{phase_end}) ---")
        
        for episode in range(phase_start, phase_end):
            target = random.choice(train_words)
            game = WordleGame(target=target, word_list=WORDS)
            
            rollout = RolloutStorage()
            guesses = []
            feedbacks_list = []
            episode_reward = 0
            
            while not game.is_over():
                turn = game.turn
                state = encode_state(guesses, feedbacks_list, turn)
                state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                
                # Get action from policy
                probs, value = model.get_action_and_value(state_tensor, word_enc)
                probs_squeezed = probs.squeeze(0)
                
                # Sample action from distribution
                dist = torch.distributions.Categorical(probs_squeezed)
                action_idx = dist.sample().item()
                log_prob = dist.log_prob(torch.tensor(action_idx).to(device))
                
                guess = WORDS[action_idx]
                feedback = game.make_guess(guess)
                guesses.append(guess)
                feedbacks_list.append(feedback)
                
                solved = game.is_solved()
                failed = game.turn >= 6 and not solved
                reward = compute_reward(feedback, solved, failed)
                episode_reward += reward
                done = solved or failed
                
                rollout.push(state, action_idx, reward,
                           value.item(), log_prob, done)
                
                if done:
                    break
            
            # Compute returns and advantages
            returns = rollout.compute_returns(gamma)
            returns = returns.to(device)
            
            values = torch.tensor(rollout.values, dtype=torch.float32).to(device)
            log_probs = torch.stack(rollout.log_probs)
            
            advantages = returns - values
            # Normalize advantages
            if len(advantages) > 1:
                advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
            
            # Policy loss (actor)
            policy_loss = -(log_probs * advantages.detach()).mean()
            
            # Value loss (critic)
            value_loss = F.mse_loss(values, returns)
            
            # Entropy bonus (encourages exploration)
            # Recompute for entropy
            entropy_loss = 0
            for s in rollout.states:
                s_t = torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)
                p, _ = model.get_action_and_value(s_t, word_enc)
                d = torch.distributions.Categorical(p.squeeze(0))
                entropy_loss += d.entropy()
            entropy_loss = entropy_loss / len(rollout)
            
            # Total loss
            loss = policy_loss + value_coef * value_loss - entropy_coef * entropy_loss
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            
            # Track stats
            recent_wins.append(1 if game.is_solved() else 0)
            recent_guesses.append(game.turn if game.is_solved() else 7)
            if len(recent_wins) > 1000:
                recent_wins.pop(0)
                recent_guesses.pop(0)
            
            if (episode + 1) % log_interval == 0:
                win_rate = sum(recent_wins) / len(recent_wins)
                avg_guess = sum(recent_guesses) / len(recent_guesses)
                history["episode"].append(episode + 1)
                history["reward"].append(episode_reward)
                history["win_rate"].append(win_rate)
                history["avg_guesses"].append(avg_guess)
                print(f"  Episode {episode+1}: win_rate={win_rate:.3f}, "
                      f"avg_guesses={avg_guess:.2f}")
    
    return model, history

In [ ]:
# Train A2C
print("Training A2C...")
start_time = time.time()

a2c_model, a2c_history = train_a2c(
    num_episodes=30000,       # Increase for better results (50k-100k)
    gamma=0.99,
    lr=0.001,
    entropy_coef=0.01,
    value_coef=0.5,
    log_interval=1000,
    curriculum_sizes=[100, 500, None]
)

elapsed = time.time() - start_time
print(f"\nA2C training complete in {elapsed/60:.1f} minutes")

# Save model
torch.save(a2c_model.state_dict(), "models/a2c_model.pt")
print("Saved to models/a2c_model.pt")

In [ ]:
# Plot A2C training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(a2c_history["episode"], a2c_history["win_rate"])
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Win Rate")
axes[0].set_title("A2C Win Rate")
axes[0].set_ylim(0, 1)
axes[0].axhline(y=1.0, color='r', linestyle='--', alpha=0.3, label='Optimal')
axes[0].legend()

axes[1].plot(a2c_history["episode"], a2c_history["avg_guesses"])
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Avg Guesses")
axes[1].set_title("A2C Avg Guesses")
axes[1].axhline(y=3.421, color='r', linestyle='--', alpha=0.3, label='Optimal (3.421)')
axes[1].legend()

plt.tight_layout()
plt.savefig("a2c_training.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Part 4: Evaluation — All Four Solvers

Run each solver against every word in the word list.
Measure:
- **Win rate** (solved in ≤ 6 guesses)
- **Average guesses** (for solved games)
- **Distribution** of guess counts (histogram)

In [ ]:
def evaluate_solver(solver, words, name="Solver", max_words=None):
    """
    Evaluate a solver against a list of target words.
    
    Args:
        solver: solver instance with get_guess(), update(), reset()
        words: list of target words to test
        name: display name for progress bar
        max_words: limit evaluation to first N words (None = all)
    
    Returns:
        dict with results
    """
    test_words = words[:max_words] if max_words else words
    results = []
    
    for target in tqdm(test_words, desc=name):
        game = WordleGame(target=target, word_list=words)
        solver.reset()
        
        while not game.is_over():
            guess = solver.get_guess()
            feedback = game.make_guess(guess)
            solver.update(guess, feedback)
            
            if game.is_solved():
                break
        
        guesses_used = game.turn if game.is_solved() else 7
        results.append({
            "word": target,
            "guesses": guesses_used,
            "solved": game.is_solved()
        })
    
    # Compute summary stats
    guess_counts = [r["guesses"] for r in results]
    solved_counts = [r["guesses"] for r in results if r["solved"]]
    
    summary = {
        "name": name,
        "total_words": len(test_words),
        "solved": sum(1 for r in results if r["solved"]),
        "win_rate": sum(1 for r in results if r["solved"]) / len(test_words),
        "avg_guesses": np.mean(solved_counts) if solved_counts else float('inf'),
        "avg_guesses_all": np.mean(guess_counts),
        "max_guesses": max(guess_counts),
        "min_guesses": min(guess_counts),
        "distribution": guess_counts,
        "results": results
    }
    
    return summary


def print_summary(summary):
    """Print evaluation summary."""
    print(f"\n{'='*50}")
    print(f"  {summary['name']}")
    print(f"{'='*50}")
    print(f"  Words tested:    {summary['total_words']}")
    print(f"  Solved:          {summary['solved']} ({summary['win_rate']*100:.1f}%)")
    print(f"  Avg guesses:     {summary['avg_guesses']:.3f} (solved only)")
    print(f"  Avg guesses:     {summary['avg_guesses_all']:.3f} (all, 7=fail)")
    print(f"  Min guesses:     {summary['min_guesses']}")
    print(f"  Max guesses:     {summary['max_guesses']}")
    
    # Distribution
    from collections import Counter
    dist = Counter(summary['distribution'])
    print(f"\n  Distribution:")
    for k in sorted(dist.keys()):
        label = f"{k}" if k <= 6 else "7+ (fail)"
        bar = "█" * (dist[k] * 40 // summary['total_words'])
        print(f"    {label:>10}: {dist[k]:>5} ({dist[k]/summary['total_words']*100:5.1f}%) {bar}")

In [ ]:
# ============================================================
# EVALUATE ALL FOUR SOLVERS
# ============================================================
# NOTE: Info gain solver is slow (~1-2 min per word on first guess).
# For a full run, set max_words=None. For a quick test, use max_words=50.

MAX_WORDS = None  # Set to 50 for quick test, None for full evaluation

all_results = {}

In [ ]:
# Solver 1: Frequency Heuristic
print("Evaluating Frequency Heuristic...")
freq_solver = FrequencySolver()
all_results["frequency"] = evaluate_solver(freq_solver, WORDS, "Frequency Heuristic", MAX_WORDS)
print_summary(all_results["frequency"])

In [ ]:
# Solver 2: Information Gain
# WARNING: This is slow! Expect ~30-60 min for full word list.
print("Evaluating Information Gain...")
ig_solver = InfoGainSolver()
all_results["infogain"] = evaluate_solver(ig_solver, WORDS, "Information Gain", MAX_WORDS)
print_summary(all_results["infogain"])

In [ ]:
# Solver 3: DQN
print("Evaluating DQN...")
dqn_solver = DQNSolver(model_path="models/dqn_model.pt")
all_results["dqn"] = evaluate_solver(dqn_solver, WORDS, "DQN", MAX_WORDS)
print_summary(all_results["dqn"])

In [ ]:
# Solver 4: A2C
print("Evaluating A2C...")
a2c_solver = A2CSolver(model_path="models/a2c_model.pt")
all_results["a2c"] = evaluate_solver(a2c_solver, WORDS, "A2C", MAX_WORDS)
print_summary(all_results["a2c"])

---
## Part 5: Comparison & Visualization

In [ ]:
# Comparison table
print(f"\n{'Solver':<25} {'Win Rate':>10} {'Avg Guesses':>12} {'Max':>5}")
print("-" * 55)
for key in ["frequency", "infogain", "dqn", "a2c"]:
    r = all_results[key]
    print(f"{r['name']:<25} {r['win_rate']*100:>9.1f}% {r['avg_guesses']:>12.3f} {r['max_guesses']:>5}")
print("-" * 55)
print(f"{'Optimal (Bertsimas)':<25} {'100.0%':>10} {'3.421':>12} {'5':>5}")

In [ ]:
# Histogram comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Wordle Solver Comparison: Guess Distribution", fontsize=16, fontweight='bold')

solver_keys = ["frequency", "infogain", "dqn", "a2c"]
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

for ax, key, color in zip(axes.flat, solver_keys, colors):
    r = all_results[key]
    dist = r["distribution"]
    
    bins = range(1, 9)
    counts = [dist.count(i) for i in bins]
    labels = [str(i) if i <= 6 else "7+\n(fail)" for i in bins]
    
    bars = ax.bar(labels, counts, color=color, edgecolor='white', linewidth=0.5)
    
    # Add count labels on bars
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                   str(count), ha='center', va='bottom', fontsize=9)
    
    ax.set_title(f"{r['name']}\nAvg: {r['avg_guesses']:.3f} | Win: {r['win_rate']*100:.1f}%",
               fontsize=12)
    ax.set_xlabel("Number of Guesses")
    ax.set_ylabel("Number of Words")

plt.tight_layout()
plt.savefig("solver_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Overlaid histogram for direct comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(1, 8)
width = 0.2

for i, (key, color) in enumerate(zip(solver_keys, colors)):
    r = all_results[key]
    counts = [r["distribution"].count(g) for g in range(1, 8)]
    offset = (i - 1.5) * width
    ax.bar(x + offset, counts, width, label=r["name"], color=color, edgecolor='white')

ax.set_xlabel("Number of Guesses", fontsize=12)
ax.set_ylabel("Number of Words", fontsize=12)
ax.set_title("Wordle Solver Comparison", fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([str(i) if i <= 6 else "7+ (fail)" for i in range(1, 8)])
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add optimal reference line
ax.axvline(x=3.421, color='red', linestyle='--', alpha=0.5, label='Optimal avg (3.421)')

plt.tight_layout()
plt.savefig("solver_overlay.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Hardest words for each solver
print("\nHardest words (most guesses) per solver:")
print("=" * 60)

for key in solver_keys:
    r = all_results[key]
    sorted_results = sorted(r["results"], key=lambda x: -x["guesses"])
    print(f"\n{r['name']}:")
    for item in sorted_results[:10]:
        status = "FAIL" if not item["solved"] else f"{item['guesses']} guesses"
        print(f"  {item['word'].upper():>8} — {status}")

---
## Summary

| Solver | Type | Win Rate | Avg Guesses | Reference |
|--------|------|----------|-------------|-----------|
| Frequency Heuristic | Heuristic | TBD | TBD | Baseline |
| Information Gain | Heuristic | TBD | TBD | Liu (2022) |
| DQN | Learned (RL) | TBD | TBD | Anderson & Meyer (2022) |
| A2C | Learned (RL) | TBD | TBD | Bhambri et al. (2022) |
| **Optimal** | **Exact DP** | **100%** | **3.421** | **Bertsimas & Paskov (2024)** |